# Module 09 — Evaluation

Eight modules of "does the agent work?" answered by eyeballing the event stream. Module 9 is where we stop doing that.

ADK ships an evaluation framework. You describe a conversation as a test case — input, expected tool trajectory, expected response — and the framework runs the agent, compares what actually happened against what you expected, and reports a pass/fail. Two metrics out of the box:

- **`tool_trajectory_avg_score`** — did the agent call the right tools in the right order?
- **`response_match_score`** — how close is the model's actual text to the expected text? (ROUGE-1 overlap.)

The trajectory metric is the novel one. Most eval frameworks focus on final-output quality; ADK weights *how the agent got there* equally. This is the right framing for tool-heavy agents — a correct answer via the wrong reasoning is a bug waiting to happen.

**What you'll build:**
- A `.test.json` eval file with an expected tool trajectory and expected response.
- A programmatic call to `AgentEvaluator.evaluate` that scores your agent.
- An honest look at why the default response_match threshold (0.8 ROUGE-1) fails on semantically-correct-but-worded-differently answers — and what you do about it in production (LLM-as-judge).

**Running cost:** under $0.01.

# Setup

In [ ]:
# google-adk[eval] brings in scikit-learn, rouge-score, and the eval helpers.
!pip install -q 'google-adk[eval]==1.28.0' litellm==1.83.4 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

## API Key

In [ ]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Key configured.")

## Imports

In [ ]:
import sys, warnings, asyncio, logging, tempfile, json, shutil
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

import nest_asyncio; nest_asyncio.apply()
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.evaluation import AgentEvaluator

print("✅ Imports successful.")

# The Two Metrics

| Metric | Asks | Scale | Default threshold |
|---|---|---|---|
| `tool_trajectory_avg_score` | Did the agent call the right tools in the right order with the right args? | 0–1 | 1.0 (strict) |
| `response_match_score` | How similar is the actual final response to the expected one? | 0–1 (ROUGE-1) | 0.8 |

Trajectory is strict: every expected tool call must appear with matching name and args. You'll often run with a slightly lower threshold during development, tighten as the agent matures.

Response match uses **ROUGE-1** — unigram overlap between expected and actual text. It is an embarrassingly crude metric for natural-language responses. We'll see it fail in a way that demonstrates why.

# The Agent Module

`AgentEvaluator.evaluate` expects an **importable Python module** containing a `root_agent`. We'll write that module to a temp directory and then point the evaluator at it.

In [ ]:
# Set up a temp directory to hold the agent module + eval file.
EVAL_DIR = tempfile.mkdtemp(prefix="adk_m09_")
AGENT_DIR = os.path.join(EVAL_DIR, "eval_demo_agent")
os.makedirs(AGENT_DIR, exist_ok=True)

# __init__.py makes it an importable package.
with open(os.path.join(AGENT_DIR, "__init__.py"), "w") as f:
    f.write("")

# The agent module itself. `root_agent` is the conventional name the evaluator looks for.
AGENT_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n"
    "\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n"
    "\n"
    "def get_weather(city: str) -> dict:\n"
    "    'Look up today\u2019s weather for a city.'\n"
    "    db = {\n"
    "        'Bratislava': {'city': 'Bratislava', 'condition': 'Sunny', 'temperature_c': 18},\n"
    "        'Prague': {'city': 'Prague', 'condition': 'Cloudy', 'temperature_c': 14},\n"
    "        'Munich': {'city': 'Munich', 'condition': 'Rainy', 'temperature_c': 11},\n"
    "    }\n"
    "    return db.get(city, {'error': f'No data for {city}'})\n"
    "\n"
    "root_agent = LlmAgent(\n"
    "    name='weather_agent',\n"
    "    model=LiteLlm(model='openrouter/google/gemini-2.5-flash-lite'),\n"
    "    description='Reports weather for European cities.',\n"
    "    instruction=(\n"
    "        'You report weather. Use get_weather to fetch the condition and '\n"
    "        'Celsius temperature. Answer in one sentence.'\n"
    "    ),\n"
    "    tools=[get_weather],\n"
    ")\n"
)

with open(os.path.join(AGENT_DIR, "agent.py"), "w") as f:
    f.write(AGENT_CODE)

# Make the temp dir importable.
if EVAL_DIR not in sys.path:
    sys.path.insert(0, EVAL_DIR)

print(f"✅ Agent module at {AGENT_DIR}")
print(f"   Module path: eval_demo_agent.agent")

# The Eval File — `.test.json`

The eval-set file is a JSON document with one or more test cases. Each case has a conversation (one or more user turns), each turn declares:

- **`user_content`** — what the user says.
- **`final_response`** — what you expect the model to ultimately reply with.
- **`intermediate_data.tool_uses`** — what tools you expect called, with args.
- **`session_input`** — optional; pre-populated session state to start from.

You can author these by hand, or — more practically — **run the agent in `adk web`, then click "Save as eval" on a conversation you like**. That exports this exact format. The programmatic path below is the same thing with different ergonomics.

In [ ]:
EVAL_SET = {
    "eval_set_id": "weather_basic",
    "name": "weather basic",
    "description": "One test case for the weather agent.",
    "eval_cases": [
        {
            "eval_id": "case_prague",
            "conversation": [
                {
                    "invocation_id": "inv-1",
                    "user_content": {
                        "parts": [{"text": "What is the weather in Prague?"}],
                        "role": "user",
                    },
                    "final_response": {
                        "parts": [{"text": "The weather in Prague is cloudy and 14 degrees Celsius."}],
                        "role": "model",
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {"name": "get_weather", "args": {"city": "Prague"}}
                        ],
                        "intermediate_responses": [],
                    },
                }
            ],
            "session_input": {
                "app_name": "weather_agent",
                "user_id": "tester",
                "state": {},
            },
        }
    ],
}

EVAL_FILE = os.path.join(EVAL_DIR, "weather.test.json")
with open(EVAL_FILE, "w") as f:
    json.dump(EVAL_SET, f, indent=2)

print(f"✅ Eval set written to {EVAL_FILE}")
print(f"   {len(EVAL_SET['eval_cases'])} case(s)")

# Run the Evaluation

`AgentEvaluator.evaluate()` loads the agent module, loads the eval file, runs the agent against each case, scores the result against the default thresholds, and raises `AssertionError` if anything falls below threshold.

In [ ]:
# Set a strict threshold to make the ROUGE-1 weakness visible.
# Default is 0.8; we bump to 0.95 — almost impossible for free-form text.
STRICT_CONFIG = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,
        "response_match_score": 0.95,
    }
}
with open(os.path.join(EVAL_DIR, "test_config.json"), "w") as f:
    json.dump(STRICT_CONFIG, f, indent=2)

# Clear the old module from sys.modules so changes are picked up.
for mod in list(sys.modules):
    if mod.startswith("eval_demo_agent"):
        del sys.modules[mod]

try:
    await AgentEvaluator.evaluate(
        agent_module="eval_demo_agent.agent",
        eval_dataset_file_path_or_dir=EVAL_FILE,
        num_runs=1,
    )
    print("\n✅ All eval cases passed.")
except AssertionError as e:
    print(f"\n❌ Eval failed with a strict threshold — see the table above.")

Read the output carefully. Two things will have happened.

1. **`tool_trajectory_avg_score` passed** — the agent called `get_weather({'city': 'Prague'})`, which matches the expected trajectory. Trajectory is strict; exact tool name and args must match.
2. **`response_match_score` very likely failed** — the agent produced something like *"The weather in Prague is Cloudy with a temperature of 14°C"*, the expected was *"The weather in Prague is cloudy and 14 degrees Celsius"*. ROUGE-1 overlap is ~0.63 against a threshold of 0.8. **Same information, different words.**

This is the most important thing to internalize about ADK's default eval: **trajectory testing is useful; ROUGE-1 response matching is weak.** ROUGE-1 punishes legitimate stylistic variation. Don't rely on it for production; it's a sanity check at best.

# Adjusting Thresholds — `test_config.json`

You can drop a `test_config.json` next to your test file to customize thresholds:

```json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0,
    "response_match_score": 0.5
  }
}
```

`0.5` lets through responses with only 50% unigram overlap — permissive, probably too permissive. `1.0` is "exact match" — impossible for anything longer than a few words.

**Working range in practice:** 0.6–0.7 for short responses; give up on `response_match_score` for long-form text. We'll configure a permissive threshold below and watch the test pass.

In [ ]:
CONFIG = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,   # stay strict on trajectory
        "response_match_score": 0.3,        # very permissive on text match
    }
}

with open(os.path.join(EVAL_DIR, "test_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

# Reload and re-run.
for mod in list(sys.modules):
    if mod.startswith("eval_demo_agent"):
        del sys.modules[mod]

try:
    await AgentEvaluator.evaluate(
        agent_module="eval_demo_agent.agent",
        eval_dataset_file_path_or_dir=EVAL_FILE,
        num_runs=1,
    )
    print("\n✅ All eval cases passed with the permissive threshold.")
except AssertionError as e:
    print(f"\n❌ Still failed:")
    print(str(e)[:1500])

With the threshold dropped to 0.3, the test passes. This is not a win — it's an admission that ROUGE-1 is the wrong metric. You've gained "my test turned green" and lost any signal about response quality.

For real work, replace `response_match_score` with an **LLM-as-judge** evaluator.

# The Real-World Upgrade — LLM-as-Judge

Production eval uses an LLM to grade responses. Roughly:

```python
# Pseudocode — not built into ADK today, but straightforward to wire up
def judge(prompt, expected, actual) -> float:
    judge_llm_response = judge_llm(
        f"Is this response a reasonable answer to this prompt?\n"
        f"Prompt: {prompt}\nExpected: {expected}\nActual: {actual}\n"
        f"Score 0-1 on semantic correctness. Just the number."
    )
    return float(judge_llm_response.strip())
```

ADK 1.29+ ships a Gen AI Evaluation Service integration that does this for you (Public Preview). Vertex AI users can point their ADK eval sets at it and get LLM-graded metrics alongside trajectory. Self-hosters wire the judge themselves with a second LiteLLM call.

**What matters is the shift in framing:** trajectory tells you *the agent did the right things*; LLM-judge tells you *the agent said the right thing*. Both matter; neither is ROUGE-1.

# The `adk eval` CLI Loop

Everything above ran programmatically. In daily development, you'd use the CLI instead:

```bash
# Step 1: Run the agent in the dev UI.
adk web

# ... have a conversation, click "Save as eval", enter a name.
# That writes a .test.json next to your agent.

# Step 2: Iterate.
adk eval ./my_agent ./my_agent/my_evals.test.json

# Tweak the instruction, change a tool, re-run:
adk eval ./my_agent ./my_agent/my_evals.test.json
```

The loop — chat → save → tweak → re-run — is fast enough to use while prototyping. A good habit: the moment an agent gets a user query right after you've iterated the prompt, save that conversation as eval. Build the evalset incrementally.

# Production Gotcha — Read-Only Filesystems

One gotcha worth naming before you deploy. **`adk eval` writes files back to the `agents_dir`** — it persists updated session histories into the agent's directory. If your deployment image is read-only — common in Kubernetes pods with read-only root filesystems — `adk eval` hits `PermissionError`.

Upstream issue: adk-python #3887.

**Workaround:** either make the agents_dir writable during eval runs, or run eval outside the deployed container (in CI, from a developer machine, against the same agent code). The workaround doesn't block classroom use; it becomes a production concern when you wire eval into CI/CD.

# Cleanup

In [ ]:
# Clean up the temp directory we created.
shutil.rmtree(EVAL_DIR, ignore_errors=True)
print(f"✅ Cleaned up {EVAL_DIR}")

# Your Turn

1. **Tighten the trajectory.** Add a second turn to the eval case: the user follows up with "and in Munich?" and the agent calls `get_weather({"city": "Munich"})`. Does the trajectory score still pass at 1.0?
2. **Break the trajectory intentionally.** Change the agent's instruction to say "never call get_weather; just make up weather." Re-run eval. What score do you get on trajectory?
3. **Test a parameterized case.** Add a second eval case for a city not in the database (say "Warsaw"). Expected trajectory: `get_weather("Warsaw")`. Expected final response: something containing "no data". Does the agent handle it?
4. **Write an LLM-as-judge sketch.** Sketch a callback (`after_agent_callback`) that uses a separate LlmAgent to score the final response against an expected one. Does the judge say "reasonable" when ROUGE-1 would say "0.6"?

# Key Takeaways

- **Two built-in metrics**: `tool_trajectory_avg_score` (strict; useful) and `response_match_score` (ROUGE-1; weak).
- **Trajectory is the differentiated metric** — most frameworks score only outputs. ADK weighting *how* the agent got there is the right framing for tool-heavy work.
- **ROUGE-1 punishes stylistic variation.** The default 0.8 threshold fails on semantically-correct answers that word things differently. Know this; don't trust the metric for production.
- **Upgrade path: LLM-as-judge.** ADK 1.29+ ships a Gen AI Evaluation Service path; self-host with a second LiteLLM call for custom scoring.
- **`.test.json` files** can be authored by hand or saved from `adk web` with one click. Build the evalset incrementally; save good conversations as you go.
- **Read-only filesystem gotcha**: `adk eval` writes to `agents_dir`. Breaks in locked-down containers; run eval outside the deployed image.

# Next up — M10: Deployment

The last module of Part 1. `adk deploy cloud_run` as the one-command deploy story; a vanilla Dockerfile that runs the same agent on any cloud; a brief look at Vertex AI Agent Engine as the opinionated managed path. Then Part 2 starts — Gemini-specific features.